# 01 Prepare WikiArt Subset

Use this notebook after downloading and extracting the WikiArt Kaggle archive.

Expected files:

```text
data/raw/wikiart/classes.csv
data/raw/wikiart/wclasses.csv
data/raw/wikiart/**/*.jpg
```

Outputs:

```text
data/processed/wikiart_subset/wikiart_subset_metadata.csv
data/labels/critique_labels_v1.csv
```

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

RAW_DIR = ROOT / 'data' / 'raw' / 'wikiart'
PROCESSED_DIR = ROOT / 'data' / 'processed' / 'wikiart_subset'
LABELS_DIR = ROOT / 'data' / 'labels'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)

classes_path = RAW_DIR / 'classes.csv'
wclasses_path = RAW_DIR / 'wclasses.csv'

print('Project root:', ROOT)
print('Classes exists:', classes_path.exists())
print('Weighted classes exists:', wclasses_path.exists())

## Load Metadata

`classes.csv` should contain human-readable metadata like filename, artist, genre, description, width, and height. `wclasses.csv` may contain encoded artist/genre/style labels.

In [ ]:
classes = pd.read_csv(classes_path)
print(classes.shape)
classes.head()

In [ ]:
print(classes.columns.tolist())
classes.describe(include='all').T.head(20)

## Basic Cleanup

This keeps rows with a filename and useful metadata, then creates stable internal IDs.

In [ ]:
df = classes.copy()
df.columns = [c.strip().lower() for c in df.columns]

required = ['filename']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df = df.dropna(subset=['filename']).drop_duplicates(subset=['filename']).reset_index(drop=True)

for optional in ['artist', 'genre', 'description', 'subset']:
    if optional not in df.columns:
        df[optional] = ''

df['id'] = [f'ACD_{i:06d}' for i in range(1, len(df) + 1)]
df['source'] = 'wikiart'
df['source_id'] = df['filename'].astype(str).str.replace('\\\\', '/', regex=False)
df['image_path'] = 'data/raw/wikiart/' + df['filename'].astype(str).str.replace('\\\\', '/', regex=False)
df['image_url'] = ''
df['style'] = ''
df['emotion'] = ''

df[['id', 'source', 'source_id', 'image_path', 'artist', 'genre', 'description']].head()

## Select a Balanced Subset

Start with 300-500 images. This cell samples across genres where possible.

In [ ]:
TARGET_SIZE = 500
RANDOM_STATE = 42

if df['genre'].replace('', pd.NA).notna().sum() > 0:
    per_genre = max(1, TARGET_SIZE // max(df['genre'].nunique(), 1))
    subset = (
        df.groupby('genre', group_keys=False)
        .apply(lambda x: x.sample(min(len(x), per_genre), random_state=RANDOM_STATE))
        .sample(frac=1, random_state=RANDOM_STATE)
        .head(TARGET_SIZE)
        .reset_index(drop=True)
    )
else:
    subset = df.sample(min(len(df), TARGET_SIZE), random_state=RANDOM_STATE).reset_index(drop=True)

subset['id'] = [f'ACD_{i:06d}' for i in range(1, len(subset) + 1)]
print(subset.shape)
subset['genre'].value_counts().head(20)

## Create Label CSV

The critique label columns are initially blank. Fill them manually in Excel, Google Sheets, or the review notebook.

In [ ]:
label_columns = [
    'id', 'source', 'source_id', 'image_path', 'image_url', 'artist', 'style', 'genre',
    'emotion', 'description', 'composition', 'color_harmony', 'contrast', 'lighting',
    'perspective', 'critique_notes', 'split'
]

labels = subset.copy()
for col in label_columns:
    if col not in labels.columns:
        labels[col] = ''

labels['composition'] = ''
labels['color_harmony'] = ''
labels['contrast'] = ''
labels['lighting'] = ''
labels['perspective'] = ''
labels['critique_notes'] = ''

n = len(labels)
labels['split'] = ['train'] * n
labels.loc[int(n * 0.8):int(n * 0.9) - 1, 'split'] = 'validation'
labels.loc[int(n * 0.9):, 'split'] = 'test'

metadata_out = PROCESSED_DIR / 'wikiart_subset_metadata.csv'
labels_out = LABELS_DIR / 'critique_labels_v1.csv'

subset.to_csv(metadata_out, index=False)
labels[label_columns].to_csv(labels_out, index=False)

print('Wrote:', metadata_out)
print('Wrote:', labels_out)
labels[label_columns].head()